# Amazon Nova RFT Single-Turn Training - Quick Start Guide

> **Note**: This notebook uses the legacy `NovaModelCustomizer` facade, which is
> deprecated and will be removed in a future release. For new projects, use
> `ForgeTrainer` and `ForgeEvaluator` directly — see `nova_quickstart.ipynb`
> and `rft_multiturn_quickstart.ipynb` for examples of the recommended API.

This notebook walks through Reinforcement Fine-Tuning (RFT) for Nova models.

## Table of Contents

1. [Step 1: Install and Import](#step-1-install-and-import)
2. [Step 2: Configure Your AWS Resources](#step-2-configure-your-aws-resources)
3. [Step 3: Create Training Reward Function](#step-3-create-training-reward-function)
4. [Step 4: Prepare RFT Training Dataset](#step-4-prepare-rft-training-dataset)
5. [Step 5: Configure Runtime](#step-5-configure-runtime)
6. [Step 6: Deploy and Validate Reward Lambda](#step-6-deploy-and-validate-reward-lambda)
7. [Step 7: Start Training](#step-7-start-training)
8. [Step 8: Monitor Training Progress](#step-8-monitor-training-progress)
9. [Step 9: Run Evaluation](#step-9-run-evaluation)
10. [Step 10: Deploy Your Model](#step-10-deploy-your-model)

## Prerequisites

- AWS credentials configured
- S3 bucket for data and model artifacts
- IAM permissions for SageMaker, Lambda, and Bedrock
- Nova Forge SDK installed

## What is RFT?

Reinforcement Fine-Tuning (RFT) uses reward functions to guide model training:
- A reward function scores model outputs
- The model is optimized to maximize reward scores
- Supports both training and evaluation modes

## Step 1: Install and Import

In [ ]:
!cd ../ && pip install .

In [ ]:
import json

import boto3

from amzn_nova_forge import *

print("✅ SDK imported successfully!")

## Step 2: Configure Your AWS Resources

In [ ]:
S3_BUCKET = "your-bucket-name"  # TODO: Replace
S3_PREFIX = "rft-singleturn-demo"
S3_DATA_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/input"
S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

print(f"Data Path: {S3_DATA_PATH}")
print(f"Output Path: {S3_OUTPUT_PATH}")

## Step 3: Create Training Reward Function

The reward function receives a list of samples and must return:
- `id`: Sample identifier
- `aggregate_reward_score`: Numeric reward score

In [ ]:
training_reward_code = """
from dataclasses import asdict, dataclass

@dataclass
class RewardOutput:
    id: str
    aggregate_reward_score: float

def lambda_handler(event, context):
    scores = []
    for sample in event:
        sample_id = sample.get(\"id\", \"unknown\")
        messages = sample.get(\"messages\", [])
        reference = sample.get(\"reference_answer\", \"\")

        completion = \"\"
        for msg in reversed(messages):
            if msg.get(\"role\") in [\"assistant\", \"nova_assistant\"]:
                completion = msg.get(\"content\", \"\")
                break

        if completion.strip().lower() == reference.strip().lower():
            reward = 1.0
        elif reference.lower() in completion.lower():
            reward = 0.5
        else:
            reward = -1.0

        scores.append(RewardOutput(id=sample_id, aggregate_reward_score=reward))

    return [asdict(s) for s in scores]
"""

with open("rft_training_reward.py", "w") as f:
    f.write(training_reward_code)

print("✅ Training reward function created: rft_training_reward.py")

## Step 4: Prepare RFT Training Dataset

RFT datasets must include:
- `id`: Unique identifier
- `messages`: Conversation in Converse format
- `reference_answer`: Ground truth (optional but recommended)

In [ ]:
rft_training_data = [
    {
        "id": f"sample_{i}",
        "messages": [
            {"role": "user", "content": "What is machine learning?"},
            {
                "role": "assistant",
                "content": "Machine learning is a subset of AI that enables systems to learn from data.",
            },
        ],
        "reference_answer": "Machine learning is a subset of artificial intelligence.",
    }
    for i in range(200)
]
rft_training_data.extend(
    [
        {
            "id": f"sample_{200 + i}",
            "messages": [
                {"role": "user", "content": "Explain AWS in one sentence."},
                {"role": "assistant", "content": "AWS is a cloud computing platform."},
            ],
            "reference_answer": "AWS is Amazon's cloud computing platform.",
        }
        for i in range(200)
    ]
)

with open("rft_training_data.jsonl", "w") as f:
    for item in rft_training_data:
        f.write(json.dumps(item) + "\n")

print(f"✅ Created {len(rft_training_data)} training samples")

In [ ]:
loader = JSONLDatasetLoader()
loader.load("rft_training_data.jsonl")
loader.show(n=2)

rft_train_s3_path = loader.save(f"{S3_DATA_PATH}/rft_train.jsonl")
print(f"\n✅ Training data uploaded to: {rft_train_s3_path}")

## Step 5: Configure Runtime

Pass `rft_lambda` to the runtime manager — it accepts either a local `.py` file path or an existing Lambda ARN.
The runtime uses this for both training (reward function) and evaluation.

### Lambda Naming Requirements
- **SMHP**: Function name must contain `SageMaker` (exact casing), e.g. `my-SageMaker-reward`
- **SMTJ**: No naming restrictions

In [ ]:
LAMBDA_NAME = "my-SageMaker-rft-reward"  # TODO: Update — must contain 'SageMaker' for SMHP

# SMHP runtime — pass rft_lambda as the local .py file path
runtime = SMHPRuntimeManager(
    instance_type="ml.p5.48xlarge",
    instance_count=4,
    cluster_name="your-cluster-name",  # TODO: Replace
    namespace="your-namespace",  # TODO: Replace
    rft_lambda="rft_training_reward.py",
)

# For SMTJ, use:
# runtime = SMTJRuntimeManager(
#     instance_type="ml.p5.48xlarge",
#     instance_count=4,
#     rft_lambda="rft_training_reward.py",
# )

# Alternatively, if you already have a deployed Lambda ARN:
# runtime = SMHPRuntimeManager(
#     ...,
#     rft_lambda="arn:aws:lambda:us-east-1:123456789012:function:my-SageMaker-rft-reward",
# )

print(f"✅ Runtime configured: {runtime.instance_type} x{runtime.instance_count}")

## Step 6: Validate and Deploy Reward Lambda

First validate the local `.py` file by running `lambda_handler` directly (no deployment needed).
Then deploy to AWS Lambda — `runtime.rft_lambda_arn` is set automatically after deploy.

Skip deploy if you passed an ARN directly to `rft_lambda` above.

In [ ]:
# Validate the local reward function against sample training data before deploying
# Executes lambda_handler directly — no Lambda ARN required
runtime.validate_lambda(
    data_s3_path=rft_train_s3_path,
    validation_samples=5,
)

print("✅ Lambda validation passed")

In [ ]:
# Deploy the reward function — defaults to runtime.rft_lambda as source
LAMBDA_ARN = runtime.deploy_lambda(lambda_name=LAMBDA_NAME)

print(f"✅ Lambda deployed: {LAMBDA_ARN}")
print(f"   runtime.rft_lambda_arn = {runtime.rft_lambda_arn}")

## Step 7: Start Training

`runtime.rft_lambda_arn` is automatically used by `train()` — no need to pass it explicitly.

In [ ]:
# DEPRECATED: Use ForgeTrainer instead of NovaModelCustomizer for new projects.
# Example: trainer = ForgeTrainer(model=..., method=..., infra=...,
#              training_data_s3_path=..., config=ForgeConfig(output_s3_path=...))
rft_customizer = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.RFT_LORA,
    infra=runtime,
    data_s3_path=rft_train_s3_path,
    output_s3_path=f"{S3_OUTPUT_PATH}/rft-training",
    validation_config={"iam": True, "infra": True},
)

training_result = rft_customizer.train(
    job_name="rft-singleturn-training",
    overrides={
        "max_steps": 100,
        "save_steps": 30,
        "lr": 5e-6,
        "global_batch_size": 64,
    },
)

print(f"🚀 RFT Training started!")
print(f"   Job ID: {training_result.job_id}")
print(f"   Checkpoint: {training_result.model_artifacts.checkpoint_s3_path}")
print(f"   Output: {training_result.model_artifacts.output_s3_path}")

job_id = training_result.job_id

In [ ]:
training_result.dump("training_result.json")
print("✅ Training result saved")

In [ ]:
loaded_training_result = TrainingResult.load("training_result.json")
print(f"✅ Loaded — Job ID: {loaded_training_result.job_id}")

## Step 8: Monitor Training Progress

In [ ]:
rft_customizer.get_logs(limit=50, start_from_head=False)

In [ ]:
# After training completes
monitor = CloudWatchLogMonitor.from_job_result(job_result=loaded_training_result)
monitor.show_logs(limit=100, start_from_head=True)

## Step 9: Run Evaluation

The same `runtime` is reused for evaluation. `reward_lambda_arn` is automatically passed to `RFT_EVAL` — no need to pass it explicitly.
You can also create another `runtime` for evaluation if you want to use another lambda or override in `evaluate(processor={"lambda_arn": EVAL_LAMBDA_ARN})`

In [ ]:
eval_data = [
    {
        "id": f"eval_{i}",
        "messages": [
            {"role": "user", "content": "What is cloud computing?"},
            {
                "role": "assistant",
                "content": "Cloud computing delivers computing services over the internet.",
            },
        ],
        "reference_answer": "Cloud computing provides on-demand computing resources.",
    }
    for i in range(50)
]

with open("rft_eval_data.jsonl", "w") as f:
    for item in eval_data:
        f.write(json.dumps(item) + "\n")

eval_loader = JSONLDatasetLoader()
eval_loader.load("rft_eval_data.jsonl")
eval_s3_path = eval_loader.save(f"{S3_DATA_PATH}/rft_eval.jsonl")
print(f"✅ Evaluation data uploaded to: {eval_s3_path}")

In [ ]:
# DEPRECATED: Use ForgeEvaluator instead of NovaModelCustomizer for new projects.
# Example: evaluator = ForgeEvaluator(model=..., infra=..., data_s3_path=...,
#              config=ForgeConfig(output_s3_path=...))
evaluator = NovaModelCustomizer(
    model=Model.NOVA_LITE_2,
    method=TrainingMethod.EVALUATION,
    infra=runtime,  # reuse the same runtime — rft_lambda_arn is already set
    data_s3_path=eval_s3_path,
    output_s3_path=f"{S3_OUTPUT_PATH}/rft-evaluation",
    validation_config={"iam": True, "infra": True},
)

# RFT_EVAL: runtime.rft_lambda_arn is auto-used as rl_env.reward_lambda_arn
eval_result = evaluator.evaluate(
    job_name="rft-singleturn-eval",
    eval_task=EvaluationTask.RFT_EVAL,
    data_s3_path=eval_s3_path,
    # model_path=training_result.model_artifacts.checkpoint_s3_path,  # use trained model
)

print(f"🚀 Evaluation started!")
print(f"   Job ID: {eval_result.job_id}")
print(f"   Output: {eval_result.eval_output_path}")

## Step 10: Deploy Your Model

In [ ]:
# DEPRECATED: Use ForgeDeployer instead of NovaModelCustomizer.deploy() for new projects.
# Example: deployer = ForgeDeployer(region="us-east-1", model=Model.NOVA_LITE_2)
#          deployer.deploy(model_artifact_path=..., deploy_platform=DeployPlatform.SAGEMAKER, ...)

# deployment_result = rft_customizer.deploy(
#     deploy_platform=DeployPlatform.SAGEMAKER,
#     unit_count=1,
#     endpoint_name="rft-singleturn-model",
#     sagemaker_environment=SageMakerEndpointEnvironment(
#         context_length=8000,
#         max_concurrency=4,
#     ),
#     job_result=training_result,
# )
# print(f"Endpoint: {deployment_result.endpoint.endpoint_name}")

## Resources

- [AWS RFT Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/nova-hp-rft-nova2.html)
- SDK Spec: `docs/spec.md`
- RFT Multi-turn: `docs/rft_multiturn.md`